# Paso 4 · Preprocesamiento y modelado con PySpark

**Qué hace este notebook:** repite lo del notebook 3 pero con PySpark: mismas variables, **misma partición** (leída desde Drive por `id`), mismos seis modelos y espacios de búsqueda, con `ParamGridBuilder` + `CrossValidator(numFolds=3)` y `areaUnderROC` como criterio.

**Reglas del enunciado que se respetan**
- Dataset completo leído como CSV con Spark. Prohibido `.toPandas()` o `.collect()` sobre el dataset completo.
- Nada de `randomSplit`: la partición se lee del archivo `split_80_20_seed42.parquet`.
- Transformadores ajustados **solo con train**.
- `.persist(MEMORY_AND_DISK)` del DataFrame **después** del `VectorAssembler` y antes de entrenar.
- Búsqueda de hiperparámetros solo con `ParamGridBuilder` + `CrossValidator`, sin bucles `for`.
- Tras entrenar, solo se traen al driver `id`, `default` y la puntuación del conjunto de prueba (el tiempo de esa transferencia se registra aparte).

**Importante:** Spark es *perezoso*: no ejecuta nada hasta que hay una acción (`count`, `fit`, ...). Por eso el tiempo de "carga" se mide al materializar los datos.

## 1. Instalar Java y PySpark (solo la primera vez en cada sesión de Colab)

In [ ]:
!apt-get install -y -qq openjdk-17-jdk-headless > /dev/null 2>&1
!pip install -q pyspark pyarrow scikit-learn
!java -version

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

## 2. Configuración de Spark
El enunciado pide 8 GB de memoria para el driver y 8 GB para el executor. En Colab Spark corre en **modo local**: driver y executor son **la misma JVM**, por lo que `spark.executor.memory` no tiene efecto y la memoria útil es la del driver. Por eso asignamos al driver la suma (16 GB). Es la configuración "equivalente debidamente justificada" que permite el enunciado.

Además se desactiva `spark.sql.ansi.enabled` para que un valor no numérico al convertir (`cast`) se vuelva nulo en lugar de lanzar un error (así se comportan también pandas y scikit-learn).

In [ ]:
import os, glob, time, json, shutil, gzip
import numpy as np, pandas as pd, psutil

p_java = '/usr/lib/jvm/java-17-openjdk-amd64'
if os.path.exists(p_java):
    os.environ['JAVA_HOME'] = p_java

from pyspark import StorageLevel
from pyspark.sql import SparkSession, functions as F
from pyspark.ml import Pipeline
from pyspark.ml.feature import Imputer, StringIndexer, OneHotEncoder, VectorAssembler, StandardScaler
from pyspark.ml.classification import (LogisticRegression, DecisionTreeClassifier, RandomForestClassifier,
                                       GBTClassifier, LinearSVC, NaiveBayes)
from pyspark.ml.tuning import ParamGridBuilder, CrossValidator
from pyspark.ml.evaluation import BinaryClassificationEvaluator
from pyspark.ml.functions import vector_to_array

RUTA = '/content/drive/MyDrive/ML_LendingClub'   # <- ajusta si tu carpeta se llama distinto
LOCAL_DIR = '/content'                            # disco local de Colab (más rápido que Drive para leer el CSV)
SEMILLA = 42
DRIVER_MEM = '16g'      # 8g driver + 8g executor (modo local: una sola JVM)
NPART = 32              # particiones del DataFrame ya preprocesado (4 por núcleo)
PARALELISMO_CV = 2      # modelos evaluados en paralelo por CrossValidator
MULTILINE = False       # pon True solo si la verificación de conteos de la sección 4 falla
RAPIDO = False          # solo para pruebas: True usa rejillas mínimas. Déjalo en False para la corrida real

DIR_RES = RUTA + '/resultados_pyspark'
os.makedirs(DIR_RES, exist_ok=True)

spark = (SparkSession.builder
         .appName('LendingClub_Optimized')
         .master('local[*]')
         .config('spark.sql.shuffle.partitions', '400')
         .config('spark.default.parallelism', '400')
         .config('spark.executor.memory', '8g')
         .config('spark.driver.memory', DRIVER_MEM)
         .config('spark.memory.fraction', 0.8)
         .config('spark.memory.storageFraction', 0.3)
         .config('spark.sql.ansi.enabled', 'false')
         .config('spark.sql.execution.arrow.pyspark.enabled', 'true')
         .getOrCreate())

HW = {'nucleos': os.cpu_count(), 'ram_gb': round(psutil.virtual_memory().total / 1e9, 1)}
print('Spark', spark.version, '| hardware declarado:', HW)
print('Memoria del driver:', spark.sparkContext.getConf().get('spark.driver.memory'))

## 3. Copiar el CSV al disco local de Colab
Leer un archivo de 1,7 GB directamente desde Drive es lento y poco estable. Lo copiamos una vez al disco local de la sesión (el tiempo se registra aparte). Usamos el **CSV sin comprimir**: un `.gz` no se puede repartir entre núcleos y Spark lo leería en un solo hilo.

In [ ]:
candidatos = [f for f in glob.glob(RUTA + '/**/accepted_2007_to_2018Q4.csv', recursive=True) if os.path.isfile(f)]
t0 = time.time()
if candidatos:
    origen = candidatos[0]
    CSV_LOCAL = os.path.join(LOCAL_DIR, 'accepted_2007_to_2018Q4.csv')
    if not os.path.exists(CSV_LOCAL):
        shutil.copyfile(origen, CSV_LOCAL)
else:   # solo existe el .gz: lo descomprimimos al disco local
    origen = sorted(f for f in glob.glob(RUTA + '/*accepted*.gz') if os.path.isfile(f))[0]
    CSV_LOCAL = os.path.join(LOCAL_DIR, 'accepted_2007_to_2018Q4.csv')
    if not os.path.exists(CSV_LOCAL):
        with gzip.open(origen, 'rb') as fi, open(CSV_LOCAL, 'wb') as fo:
            shutil.copyfileobj(fi, fo)
t_copia = time.time() - t0
print(f'Origen: {origen}\nCSV local: {CSV_LOCAL} ({os.path.getsize(CSV_LOCAL)/1e9:.2f} GB) | copia: {t_copia:,.1f} s')

## 4. Leer con Spark, filtrar y unir la partición común
Mismas columnas, mismo filtro (*Fully Paid* y *Charged Off*) y mismo `default` que en scikit-learn. La partición se une por `id` con el archivo de Drive (una tabla pequeña, por eso se usa `broadcast`).

**Verificación clave:** los conteos deben coincidir con los de pandas (1.076.751 y 268.559). Si Spark partiera mal alguna fila con saltos de línea dentro de un campo de texto, los conteos no coincidirían y lo veríamos aquí.

In [ ]:
NUM = ['loan_amnt', 'int_rate', 'installment', 'annual_inc', 'dti', 'fico_range_high', 'delinq_2yrs',
       'inq_last_6mths', 'open_acc', 'pub_rec', 'revol_bal', 'revol_util', 'total_acc', 'mort_acc',
       'pub_rec_bankruptcies']
CAT = ['purpose', 'home_ownership', 'addr_state', 'verification_status']
ESPERADO = {'Fully Paid': 1076751, 'Charged Off': 268559}

t0 = time.time()
raw = (spark.read.option('header', True).option('quote', '"').option('escape', '"')
       .option('multiLine', MULTILINE).csv(CSV_LOCAL))

faltan = [c for c in ['id', 'loan_status', 'term', 'emp_length'] + NUM + CAT if c not in raw.columns]
assert not faltan, f'Columnas que no existen en el archivo: {faltan}'

sel = raw.select('id', 'loan_status', 'term', 'emp_length',
                 *[F.col(c).cast('double').alias(c) for c in NUM], *CAT)
cerrados = sel.filter(F.col('loan_status').isin('Fully Paid', 'Charged Off'))

split_df = spark.read.parquet(RUTA + '/split_80_20_seed42.parquet')
base = cerrados.join(F.broadcast(split_df), on='id', how='left')
base = base.persist(StorageLevel.MEMORY_AND_DISK)
n_base = base.count()            # aquí se lee y se parsea el CSV completo
t_carga = time.time() - t0

conteos = {r['loan_status']: r['count'] for r in base.groupBy('loan_status').count().collect()}
sin_split = base.filter(F.col('split').isNull()).count()
print(f'Carga + filtro + unión con la partición: {t_carga:,.1f} s | filas: {n_base:,}')
print('Conteos por estado:', conteos)
print('Filas sin asignación train/test:', sin_split)
if not RAPIDO:
    assert conteos == ESPERADO, 'Los conteos no coinciden con pandas: prueba MULTILINE=True y vuelve a ejecutar desde la sección 2'
assert sin_split == 0, 'Hay ids sin partición: avísame'

## 5. Preprocesamiento (ajustado solo con train)
Mismos pasos que en scikit-learn:
- `term` y `emp_length` a número; categóricas nulas → `'Desconocido'`.
- Numéricas: `Imputer` con la **mediana de train** + `StandardScaler` (con media y desviación).
- Categóricas: `StringIndexer` + `OneHotEncoder(dropLast=False)`, que equivale al one-hot completo de scikit-learn.
- `VectorAssembler` final y **`persist`** (obligatorio).

Diferencias menores que hay que mencionar en el informe: la mediana de `Imputer` es aproximada (scikit-learn la calcula exacta) y el `StandardScaler` de Spark usa desviación muestral (n-1) frente a la poblacional (n) de scikit-learn; con más de un millón de filas el efecto es despreciable.

In [ ]:
t0 = time.time()

datos = (base
         .withColumn('term_meses', F.regexp_extract('term', r'(\d+)', 1).cast('double'))
         .withColumn('emp_length_anios',
                     F.when(F.col('emp_length') == '< 1 year', F.lit(0.0))
                      .otherwise(F.regexp_extract('emp_length', r'(\d+)', 1).cast('double')))
         .withColumn('default', F.when(F.col('loan_status') == 'Charged Off', 1.0).otherwise(0.0)))
for c in CAT:
    datos = datos.withColumn(c, F.coalesce(F.col(c), F.lit('Desconocido')))

NUM_COLS = NUM + ['term_meses', 'emp_length_anios']
imputer = Imputer(strategy='median', inputCols=NUM_COLS, outputCols=[c + '_imp' for c in NUM_COLS])
indexers = [StringIndexer(inputCol=c, outputCol=c + '_idx', handleInvalid='keep') for c in CAT]
ohe = OneHotEncoder(inputCols=[c + '_idx' for c in CAT], outputCols=[c + '_ohe' for c in CAT],
                    dropLast=False, handleInvalid='keep')
va_num = VectorAssembler(inputCols=[c + '_imp' for c in NUM_COLS], outputCol='num_vec')
scaler = StandardScaler(inputCol='num_vec', outputCol='num_esc', withMean=True, withStd=True)
va_final = VectorAssembler(inputCols=['num_esc'] + [c + '_ohe' for c in CAT], outputCol='features')

datos_train = datos.filter(F.col('split') == 'train')
datos_test = datos.filter(F.col('split') == 'test')

pipe_model = Pipeline(stages=[imputer] + indexers + [ohe, va_num, scaler, va_final]).fit(datos_train)   # solo train

train_p = pipe_model.transform(datos_train).select('id', 'default', 'features').repartition(NPART)
test_p = pipe_model.transform(datos_test).select('id', 'default', 'features').repartition(NPART)
train_p = train_p.persist(StorageLevel.MEMORY_AND_DISK)      # cache obligatorio tras el VectorAssembler
test_p = test_p.persist(StorageLevel.MEMORY_AND_DISK)
n_tr, n_te = train_p.count(), test_p.count()
t_prep = time.time() - t0

try:
    pipe_model.write().overwrite().save(DIR_RES + '/preprocesador')
except Exception as e:
    print('No se pudo guardar el preprocesador (no es grave):', e)

print(f'Preprocesamiento + materialización: {t_prep:,.1f} s')
print(f'train: {n_tr:,} | test: {n_te:,}')
print('Tamaño del vector de características:', len(train_p.first()['features']))

## 6. Modelos y espacios de búsqueda
Mismos seis modelos y rejillas que en scikit-learn. Decisiones de equivalencia que hay que **documentar**:
- Regularización L2: `elasticNetParam=0` en regresión logística. `regParam ∈ {1e-6, 1e-5, 1e-4}` (en scikit-learn `C = 1/(regParam·n)`).
- `standardization=False` en regresión logística y LinearSVC: los datos ya vienen escalados y así la regularización actúa igual que en scikit-learn.
- `LinearSVC` de Spark solo implementa la pérdida *hinge*. En scikit-learn la versión final usa `squared_hinge` primal, porque con `hinge` dual se obtuvo un modelo degenerado (AUC 0.515); es una diferencia real entre entornos que se documenta en el informe.
- Árboles: `maxBins=32` (valor por defecto). Spark discretiza cada variable continua en un máximo de 32 intervalos; scikit-learn evalúa cortes exactos.
- `BinaryClassificationEvaluator(numBins=0)`: AUC exacto, sin la aproximación de 1000 bins que Spark usa por defecto.
- **Columna de puntuación del evaluador:** se usa la `probability` de la clase positiva (y `rawPrediction` solo en LinearSVC, que no produce probabilidades). La columna `rawPrediction` por defecto no sirve para ordenar en todos los modelos: en el árbol de decisión son conteos de entrenamiento por hoja (dependen del tamaño de la hoja, no de su pureza) y en Naive Bayes son log-verosimilitudes sin normalizar. Con ella el AUC saldría incorrecto, y también la selección del `CrossValidator`. Es una diferencia de implementación que conviene discutir en el informe.
- Regresión logística y LinearSVC con `maxIter=100` (el valor por defecto de Spark).
- Naive Bayes gaussiano, sin búsqueda de hiperparámetros.

In [ ]:
def g(normal, rapido):
    return rapido if RAPIDO else normal

REGS = [1e-6, 1e-5, 1e-4]
kw = dict(featuresCol='features', labelCol='default')

lr = LogisticRegression(elasticNetParam=0.0, maxIter=100, standardization=False, **kw)
dt = DecisionTreeClassifier(maxBins=32, seed=SEMILLA, **kw)
rf = RandomForestClassifier(maxBins=32, seed=SEMILLA, **kw)
gbt = GBTClassifier(stepSize=0.1, maxBins=32, seed=SEMILLA, **kw)
svc = LinearSVC(maxIter=100, standardization=False, **kw)
nb = NaiveBayes(modelType='gaussian', **kw)

grid_lr = ParamGridBuilder().addGrid(lr.regParam, g(REGS, REGS[:1])).build()
grid_dt = ParamGridBuilder().addGrid(dt.maxDepth, g([5, 10, 15], [3])).build()
grid_svc = ParamGridBuilder().addGrid(svc.regParam, g(REGS, REGS[:1])).build()
grid_gbt = (ParamGridBuilder().addGrid(gbt.maxIter, g([50, 100], [5]))
            .addGrid(gbt.maxDepth, g([3, 5], [3])).build())
grid_rf = (ParamGridBuilder().addGrid(rf.numTrees, g([10, 50, 100], [5]))
           .addGrid(rf.maxDepth, g([5, 10, 15], [3])).build())

MODELOS = {   # nombre: (estimador, mapas de parámetros o None)  -- de lo más rápido a lo más lento
    'NaiveBayes':         (nb, None),
    'LogisticRegression': (lr, grid_lr),
    'DecisionTree':       (dt, grid_dt),
    'LinearSVC':          (svc, grid_svc),
    'GradientBoosting':   (gbt, grid_gbt),
    'RandomForest':       (rf, grid_rf),
}
def crear_evaluador(nombre):
    col = 'rawPrediction' if nombre == 'LinearSVC' else 'probability'
    return BinaryClassificationEvaluator(labelCol='default', rawPredictionCol=col,
                                         metricName='areaUnderROC', numBins=0)

print('Modelos:', list(MODELOS))

## 7. Entrenar, evaluar y guardar (con puntos de control)
- **Tiempo de entrenamiento** = `CrossValidator.fit` completo (todas las combinaciones × 3 pliegues + reajuste final).
- **Tiempo de predicción** = `transform` sobre test forzado con una acción (`count`).
- **Transferencia al driver** (solo `id`, `default` y puntuación de test) se mide **aparte**.
- Puntuación: `probability[1]`, o `rawPrediction[1]` en LinearSVC. Umbral: 0,5 y 0 respectivamente.
- Como control, el AUC de Spark se recalcula con scikit-learn sobre las puntuaciones guardadas: deben coincidir.

Si Colab se desconecta, vuelve a ejecutar todo: los modelos terminados se saltan solos (hay que repetir las secciones 3 a 5, que son rápidas comparadas con el entrenamiento).

In [ ]:
from sklearn.metrics import roc_auc_score

def a_nativo(v):
    return v.item() if hasattr(v, 'item') else v

def correr_modelo(nombre, estimador, mapas):
    ruta_json = f'{DIR_RES}/{nombre}.json'
    if os.path.exists(ruta_json):
        print(f'[{nombre}] ya estaba terminado, se salta.')
        return
    print(f'[{nombre}] entrenando...')
    evaluador = crear_evaluador(nombre)
    t0 = time.time()
    if mapas:
        cv = CrossValidator(estimator=estimador, estimatorParamMaps=mapas, evaluator=evaluador,
                            numFolds=3, seed=SEMILLA, parallelism=PARALELISMO_CV)
        cv_model = cv.fit(train_p)
        modelo = cv_model.bestModel
        k = int(np.argmax(cv_model.avgMetrics))
        mejores = {p.name: a_nativo(v) for p, v in mapas[k].items()}
        auc_cv = float(cv_model.avgMetrics[k])
    else:
        modelo = estimador.fit(train_p)
        mejores, auc_cv = {}, None
    t_fit = time.time() - t0

    t0 = time.time()
    pred = modelo.transform(test_p).persist(StorageLevel.MEMORY_AND_DISK)
    pred.count()
    t_pred = time.time() - t0

    auc = float(evaluador.evaluate(pred))
    conf = {(int(r['default']), int(r['prediction'])): int(r['count'])
            for r in pred.groupBy('default', 'prediction').count().collect()}
    tn, fp, fn, tp = conf.get((0, 0), 0), conf.get((0, 1), 0), conf.get((1, 0), 0), conf.get((1, 1), 0)
    precision = tp / (tp + fp) if (tp + fp) else 0.0
    recall = tp / (tp + fn) if (tp + fn) else 0.0
    f1 = 2 * precision * recall / (precision + recall) if (precision + recall) else 0.0
    accuracy = (tp + tn) / (tp + tn + fp + fn)

    col_score = 'probability' if 'probability' in pred.columns else 'rawPrediction'
    t0 = time.time()
    sc = pred.select('id', 'default', vector_to_array(col_score)[1].alias('score')).toPandas()
    t_transf = time.time() - t0
    auc_chequeo = float(roc_auc_score(sc['default'], sc['score']))

    res = {
        'modelo': nombre, 'mejores_params': mejores, 'auc_cv': auc_cv,
        'accuracy': accuracy, 'precision': precision, 'recall': recall, 'f1': f1, 'auc': auc,
        'auc_chequeo_sklearn': auc_chequeo,
        'tn': tn, 'fp': fp, 'fn': fn, 'tp': tp,
        'umbral': 0.5 if col_score == 'probability' else 0.0,
        't_entrenamiento_s': round(t_fit, 1), 't_prediccion_s': round(t_pred, 2),
        't_transferencia_s': round(t_transf, 2),
        't_preprocesamiento_s': round(t_prep, 1), 't_carga_s': round(t_carga, 1), 't_copia_csv_s': round(t_copia, 1),
        'hardware': HW, 'driver_mem': DRIVER_MEM, 'npart': NPART, 'paralelismo_cv': PARALELISMO_CV,
    }
    sc.to_parquet(f'{DIR_RES}/scores_{nombre}.parquet', index=False)
    try:
        modelo.write().overwrite().save(f'{DIR_RES}/modelo_{nombre}')
    except Exception as e:
        print(f'[{nombre}] no se pudo guardar el modelo (no es grave):', e)
    pred.unpersist()
    with open(ruta_json, 'w') as f:      # el json se escribe al final: es la marca de "terminado"
        json.dump(res, f, indent=1)
    print(f"[{nombre}] listo en {t_fit/60:.1f} min | AUC={auc:.4f} (chequeo sklearn {auc_chequeo:.4f}) | mejores={mejores}")

for nombre, (est, mapas) in MODELOS.items():
    correr_modelo(nombre, est, mapas)

## 8. Consolidar resultados
Tabla de métricas y tiempos, y un único archivo con las puntuaciones de todos los modelos de PySpark.

In [ ]:
filas = []
for nombre in MODELOS:
    p = f'{DIR_RES}/{nombre}.json'
    if os.path.exists(p):
        filas.append(json.load(open(p)))
resultados = pd.DataFrame(filas).set_index('modelo')
cols = ['auc', 'auc_chequeo_sklearn', 'auc_cv', 'accuracy', 'precision', 'recall', 'f1',
        't_entrenamiento_s', 't_prediccion_s', 't_transferencia_s', 'mejores_params']
resultados[cols]

In [ ]:
scores = None
for nombre in MODELOS:
    p = f'{DIR_RES}/scores_{nombre}.parquet'
    if os.path.exists(p):
        s = pd.read_parquet(p).rename(columns={'score': nombre})
        scores = s if scores is None else scores.merge(s[['id', nombre]], on='id')
scores.to_parquet(RUTA + '/scores_pyspark.parquet', index=False)
resultados[cols].to_csv(RUTA + '/resultados_pyspark.csv')
print('Guardado:', RUTA + '/scores_pyspark.parquet', scores.shape)
print('Tiempos comunes -> copia CSV: %.1f s | carga: %.1f s | preprocesamiento: %.1f s' % (t_copia, t_carga, t_prep))
print('Matrices de confusión (tn, fp, fn, tp):')
print(resultados[['tn', 'fp', 'fn', 'tp']])

## 👉 Qué copiar y pegarme en el chat
1. Salida de las secciones 2, 4 y 5 (Spark, conteos, tiempos y tamaños).
2. La tabla de resultados de la sección 8.
3. Cualquier error, completo.